In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import glob
import gc
from concurrent.futures import ProcessPoolExecutor

cesm_folder = '/glade/campaign/cesm/km-scale/archive/'
nodeep_summer = 'cam77_dyamond1_prod1'
clubb_summer = 'c124_dyamond1_prod2'
nodeep_winter = 'cam77_dyamond2_prod1'


scratch = '/glade/derecho/scratch/sarahryu/'

vars = [

    # INPUTS

    'T', 'Q', 'CLDLIQ', 'CLDICE', 'U', 'V', 'U10', 
    
    # EXTRA INPUTS
    'SST', 'LANDFRAC', 'OCNFRAC', 'ICEFRAC', 'PHIS', 'SOLIN',

    'DTCOND', 'DCQ',

    
    'DTCORE', 'DQCORE', # ADVECTION

    'MPDT', 'MPDQ', 'MPDICE', 'MPDLIQ', # MICROPHYSICS
    
    
    'EVAPPREC', 'EVAPSNOW', # SEDIMENTATION
    
    'QRL', 'QRS', # RADIATION
    
    'KVH_CLUBB', # (on interfaces)

    'SHFLX', 'LHFLX', 'QFLX',  # SURFACE FLUXES

    # EXTRAS
    'TAUX', 'TAUY', 'RAINQM', 'SNOWQM', 'GRAUQM',  'Z3', 'PMID',
    
]

stream = 'h1i' # daily accsumulated. other options are h0i, h1i, h2i

In [2]:
def make_grid(res):
    """Regular lat-lon edges for a given resolution in degrees."""
    lon_edges = np.arange(0, 360 + res, res)
    lat_edges = np.arange(-90, 90 + res, res)
    return lon_edges, lat_edges

def coarse_grain_dataset(ds, varlist, res, col_dim="ncol", area_var="area", print_vars=False):
    lon_edges, lat_edges = make_grid(res)
    nlon, nlat = len(lon_edges) - 1, len(lat_edges) - 1

    # grid-only quantities: computed once, reused for every variable
    ilon = np.clip(np.digitize(ds.lon.values, lon_edges) - 1, 0, nlon - 1)
    ilat = np.clip(np.digitize(ds.lat.values, lat_edges) - 1, 0, nlat - 1)
    flat = ilat * nlon + ilon

    w   = np.asarray(ds[area_var].values).ravel()
    den = np.bincount(flat, weights=w, minlength=nlat * nlon)

    lon_centers = 0.5 * (lon_edges[:-1] + lon_edges[1:])
    lat_centers = 0.5 * (lat_edges[:-1] + lat_edges[1:])

    def coarse_last(arr):
        """arr: (..., ncol) -> (..., nlat, nlon), any number of leading dims."""
        lead = arr.shape[:-1]
        arr2 = arr.reshape(-1, arr.shape[-1])           # (batch, ncol)
        out  = np.empty((arr2.shape[0], nlat * nlon))
        for i in range(arr2.shape[0]):
            out[i] = np.bincount(flat, weights=w * arr2[i],
                                 minlength=nlat * nlon) / den
        return out.reshape(*lead, nlat, nlon)

    results = {}
    for name in varlist:
        if print_vars:
            print(name)
        da = ds[name]
        if col_dim not in da.dims:
            print(f"skipping {name!r}: no {col_dim!r} dimension")
            continue
        other_dims = [d for d in da.dims if d != col_dim]
        da = da.transpose(*other_dims, col_dim)         # ensure ncol is last
        arr = coarse_last(da.values)

        coords = {d: ds[d].values for d in other_dims if d in ds.coords}
        coords["lat"] = lat_centers
        coords["lon"] = lon_centers
        results[name] = xr.DataArray(
            arr, dims=(*other_dims, "lat", "lon"),
            coords=coords, attrs=dict(da.attrs),
        )

    return xr.Dataset(results)



In [3]:
all_files_nodeep = glob.glob(
    os.path.join(
        cesm_folder,
        nodeep_summer,
        'atm/hist',
        f'{nodeep_summer}.cam.{stream}.*.nc'
    )
)

all_files_clubb = glob.glob(
    os.path.join(
        cesm_folder,
        clubb_summer,
        'atm/hist',
        f'{clubb_summer}.cam.{stream}.*.nc'
    )
)

In [4]:
## WINTER
all_files_nodeep = glob.glob(
    os.path.join(
        cesm_folder,
        nodeep_winter,
        'atm/hist',
        f'{nodeep_winter}.cam.{stream}.*.nc'
    )
)

## Variables


#### PTTEND AND PTEQ - for lumped approach

#### Vertical Advection / Resolved Transport
DTCORE
DQCORE

#### Microphysics
MPDT, MPDQ, MPDICE, MPDLIQ, EVAPPREC, EVAPSNOW

#### Radiative heating
QRL, QRS

#### Turbulent diffusivity (RF-diff output)
KVH_CLUBB (on interfaces)

#### Turbulent fluxes / tendencies
WPTHLP_CLUBB, WPRTP_CLUBB, DTV, STEND/RVMTEND/RCMTEND_CLUBB

#### Surface fluxes	
SHFLX, LHFLX, QFLX, TAUX, TAUY

#### State for hL, qT, qp + inputs	
T, Q, CLDLIQ, CLDICE, RAINQM, SNOWQM, GRAUQM, U, V, Z3, PMID


#### Surface / geography inputs	
SST, LANDFRAC, OCNFRAC, ICEFRAC, PHIS, SOLIN

These are the inputs the 2020 paper said you'd need to replace the distance-to-equator proxy once symmetry is gone.


#### DO WE HAVE RESOLVED-FROM-COARSE TENDENCIES? NEED THIS TO REPLICATE YOG20, otherwise i can still do it but it may be less stable.

In [ ]:
### NEW PARARELLIZED CODE

def process_one(fp):
    target_fp = os.path.join(scratch, 'leap2026/nodeep_1deg/', os.path.basename(fp))
    if os.path.exists(target_fp):
        return f"skip {target_fp}"
    ds = xr.open_dataset(fp)
    ds_1deg = coarse_grain_dataset(ds, vars, res=1.0)
    ds_1deg.to_netcdf(target_fp)
    ds.close()
    gc.collect()
    return f"done {target_fp}"


if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=5) as ex:
        for msg in ex.map(process_one, all_files_nodeep):
            print(msg)

/glade/derecho/scratch/sarahryu/tmp/ipykernel_91532/781526137.py:28: RuntimeWarning: invalid value encountered in divide
  out[i] = np.bincount(flat, weights=w * arr2[i],
/glade/derecho/scratch/sarahryu/tmp/ipykernel_91532/781526137.py:28: RuntimeWarning: invalid value encountered in divide
  out[i] = np.bincount(flat, weights=w * arr2[i],
/glade/derecho/scratch/sarahryu/tmp/ipykernel_91532/781526137.py:28: RuntimeWarning: invalid value encountered in divide
  out[i] = np.bincount(flat, weights=w * arr2[i],
/glade/derecho/scratch/sarahryu/tmp/ipykernel_91532/781526137.py:28: RuntimeWarning: invalid value encountered in divide
  out[i] = np.bincount(flat, weights=w * arr2[i],
/glade/derecho/scratch/sarahryu/tmp/ipykernel_91532/781526137.py:28: RuntimeWarning: invalid value encountered in divide
  out[i] = np.bincount(flat, weights=w * arr2[i],
/glade/derecho/scratch/sarahryu/tmp/ipykernel_91532/781526137.py:28: RuntimeWarning: invalid value encountered in divide
  out[i] = np.bincount(f

In [ ]:
###OLD CODE

for fp in all_files_nodeep:
    target_fp = os.path.join(scratch, 'leap2026/nodeep_1deg/', os.path.basename(fp))
    print(target_fp)z
    if os.path.exists(target_fp):
        continue

    ds = xr.open_dataset(fp)

    ds_1deg = coarse_grain_dataset(ds, vars, res=1.0, print_vars=True)
    ds_1deg.to_netcdf(target_fp)

    gc.collect()

In [ ]:
fp = all_files_nodeep[0]

ds = xr.open_dataset(fp)

for var in vars:
    if var not in ds.data_vars:
        print(f"{var!r} missing")